# Music genre classification

Compare an RBF SVM and random forest using FMA-small's provided splits.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

root = Path.cwd() if Path("requirements.txt").is_file() else Path.cwd().parent
data = root / "data" / "fma_metadata"
results = root / "results"
results.mkdir(exist_ok=True)


## Load FMA-small

Keep the supplied split assignments and align labels with audio features by track ID.

In [ ]:
tracks = pd.read_csv(data / "tracks.csv", index_col=0, header=[0, 1], low_memory=False)
features = pd.read_csv(data / "features.csv", index_col=0, header=[0, 1, 2])
if not tracks.index.is_unique or not features.index.is_unique:
    raise ValueError("Expected unique track IDs in both source tables.")

small = tracks.loc[tracks[("set", "subset")].eq("small")]
labels = small[("track", "genre_top")]
split = small[("set", "split")]
X = features.reindex(small.index).replace([np.inf, -np.inf], np.nan)
X.columns = [" / ".join(column) for column in X.columns]
usable = labels.notna() & X.notna().any(axis=1)
print(f"Using {usable.sum()} of {len(small)} tracks.")
X, labels, split = X.loc[usable], labels.loc[usable], split.loc[usable]
assert set(split.unique()) == {"training", "validation", "test"}
genres = sorted(labels.unique())
assert len(genres) == 8
display(pd.crosstab(labels, split))


## Tune on validation data

Imputation and scaling are fitted inside each training pipeline. The search keeps the test split out and refits the chosen models on training plus validation before final scoring.

In [ ]:
development = split.ne("test")
validation = split.eq("validation")
test = split.eq("test")
training = split.eq("training")
cv = PredefinedSplit(np.where(validation.loc[development], 0, -1))

models = {
    "svm": (
        make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), SVC(kernel="rbf")),
        {"svc__C": [1, 10], "svc__gamma": ["scale", 0.001]},
    ),
    "forest": (
        make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(random_state=42)),
        {"randomforestclassifier__max_depth": [20, None],
         "randomforestclassifier__n_estimators": [150, 300]},
    ),
}

searches = {}
for name, (pipeline, parameters) in models.items():
    search = GridSearchCV(pipeline, parameters, cv=cv, scoring="f1_macro", n_jobs=2,
                          error_score="raise")
    search.fit(X.loc[development], labels.loc[development])
    searches[name] = search
    print(name, search.best_params_, round(search.best_score_, 4))


## Evaluate on the held-out test set

Each confusion matrix row shows the fraction of tracks in that true genre assigned to each predicted genre.

In [ ]:
scores = []
for name, search in searches.items():
    truth = labels.loc[test]
    prediction = search.predict(X.loc[test])
    scores.append({"model": name, "validation_macro_f1": search.best_score_,
                   "test_accuracy": accuracy_score(truth, prediction),
                   "test_macro_f1": f1_score(truth, prediction, average="macro")})

    report = classification_report(truth, prediction, labels=genres,
                                   output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(results / f"{name}_classification_report.csv")

    fig, ax = plt.subplots(figsize=(9, 7), layout="constrained")
    ConfusionMatrixDisplay.from_predictions(
        truth, prediction, labels=genres, normalize="true",
        xticks_rotation=45, values_format=".2f", ax=ax,
    )
    ax.set_title(f"{name.upper()} test confusion matrix")
    fig.savefig(results / f"{name}_confusion_matrix.png", dpi=180)
    plt.show()

metrics = pd.DataFrame(scores).set_index("model")
metrics.to_csv(results / "metrics.csv")
display(metrics)


## Optional permutation importance

The published notebook can omit running this slower diagnostic until both models are ready. Fit the selected configuration on training only before measuring feature importance on validation data.

In [ ]:
from sklearn.base import clone


def export_permutation_importance():
    for name, search in searches.items():
        model = clone(search.best_estimator_).fit(X.loc[training], labels.loc[training])
        importance = permutation_importance(
            model, X.loc[validation], labels.loc[validation],
            scoring="f1_macro", n_repeats=3, random_state=42, n_jobs=2,
        )
        pd.Series(importance.importances_mean, index=X.columns, name="macro_f1_drop").sort_values(
            ascending=False
        ).to_csv(results / f"{name}_permutation_importance.csv")


# export_permutation_importance()
